# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook provides a step-by-step workflow for loading and exploring the FAIR² dataset using the `mlcroissant` library, referencing all dataset entities (such as record sets, fields, columns) by their unique `@id` values as defined in the Croissant schema.

### Dataset Source

The dataset source is a Croissant schema accessible at:
```
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json
```

This dataset details 77 cancer survivors with second primary colorectal cancer, including clinical and pathological variables (demographics, comorbidities, cancer types, treatment history, molecular biomarkers, and more).


In [ ]:
# Ensure `mlcroissant` is available
!pip install -U mlcroissant pandas

## 1. Data Loading

Load the dataset metadata and records from the Croissant schema using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the dataset via Croissant
dataset = mlc.Dataset(croissant_url)

# Print out dataset name and description from metadata
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview

Review the available record sets and their fields (`@id` values) in the dataset. All dataset entities are referenced by their `@id`, which guarantees unique identification across schema and data.

**We enumerate all record sets in the dataset and preview their schema.**

In [ ]:
# List all available record sets and their fields by @id:
record_sets = []
for rs in dataset.record_sets:
    record_sets.append(rs["@id"])
    print(f"RecordSet @id: {rs['@id']}, Name: {getattr(rs, 'name', 'N/A')}")
    if hasattr(rs, 'fields') and rs.fields:
        for fld in rs.fields:
            print(f"  - Field @id: {fld['@id']}, Name: {getattr(fld, 'name', 'N/A')}, Data type: {getattr(fld, 'dataType', 'N/A')}")
    if hasattr(rs, 'columns') and rs.columns:
        for col in rs.columns:
            print(f"  - Column @id: {col['@id']}, Name: {getattr(col, 'name', 'N/A')} (File Column)")
if not record_sets:
    print("No record sets present in Croissant metadata. Attempting to infer from file objects...")

## 3. Data Extraction

Load data from a specific record set into a pandas DataFrame. All record set and field selection is by entity `@id`.

Here, we extract the data from the main clinical record set (replace with actual `@id` based on previous output).

In [ ]:
# Choose the primary clinical RecordSet @id
# If record_sets is empty (no top-level RecordSet), try with common conventions
if record_sets:
    main_record_set_id = record_sets[0]  # Use the first RecordSet
else:
    # Fall back to default: e.g., '/dataset/recordSets/secondPrimaryCRC' if known, or print error
    main_record_set_id = None

# List all records for each record set
dataframes = {}
if main_record_set_id is not None:
    print(f"\nExtracting records for record set '@id': {main_record_set_id}\n")
    records = list(dataset.records(record_set=main_record_set_id))
    df = pd.DataFrame(records)
    dataframes[main_record_set_id] = df
    print(f"Columns in record set {main_record_set_id}:")
    print(df.columns.tolist())
    display(df.head())
else:
    print("No RecordSet available to extract records from. Please check the Croissant metadata.")

## 4. Exploratory Data Analysis (EDA)

Perform basic data processing steps to prepare for analysis. We reference fields by their `@id` (column names derived from schema `@id`).

Common steps: filtering, normalization, grouping by categorical variables.

In [ ]:
# List numeric fields by @id:
import numpy as np
if main_record_set_id is not None:
    df = dataframes[main_record_set_id]
    # Identify numeric columns
    numeric_fields = [col for col in df.columns if np.issubdtype(df[col].dropna().dtype, np.number)]
    print("Numeric fields found:")
    print(numeric_fields)
    # Select one for demonstration
    if numeric_fields:
        numeric_field_id = numeric_fields[0]
        threshold = df[numeric_field_id].mean() if np.isfinite(df[numeric_field_id].mean()) else 0
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"\nFiltered records where {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalize
        col_norm = f"{numeric_field_id}_normalized"
        filtered_df[col_norm] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std(ddof=0)
        print(f"\nNormalized {numeric_field_id} in filtered records:")
        display(filtered_df[[numeric_field_id, col_norm]].head())

        # Find first categorical field for grouping
        category_fields = [col for col in df.columns if df[col].dtype == 'object' and len(df[col].unique()) < 20]
        group_field_id = category_fields[0] if category_fields else None
        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"\nGrouped by {group_field_id} and computed mean of {numeric_field_id}:")
            display(grouped_df.head())
    else:
        print("No numeric fields available for EDA.")
else:
    print("No RecordSet data available for EDA.")

## 5. Visualization

Visualize the distribution of a numeric variable and compare means by group (if possible).

All axes and legends reference field `@id` directly.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id is not None and 'numeric_field_id' in locals():
    # Histogram
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=15, kde=True)
    plt.title(f"Distribution of '{numeric_field_id}'")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # Boxplot by group (if available)
    if 'group_field_id' in locals() and group_field_id:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion

In this notebook, we demonstrated how to load a Croissant-based clinical dataset using `mlcroissant`, explored schema entities by `@id`, extracted records, performed EDA (including normalization and grouping), and visualized the results.

- All operations referenced record sets and fields using `@id` for full schema transparency.
- You can now extend this workflow for your own analysis or modeling, always keeping references by `@id` for maximal reproducibility and clarity.

*For more on FAIR² metadata and the Croissant format, see* [mlcommons.org/croissant](https://mlcommons.org/croissant/) *and the original dataset provider.*